# FreshBasket Sales Analysis — Day 10 Assessment Project

**Analyst:** Jordan Lee  |  **Role:** Junior Data Analyst  

**Brief:** Management handed us a messy sales export and asked for a complete analysis with recommendations. This notebook runs the full workflow: **Load → Clean → Explore → Visualize → SQL → Insights → Report**.

> Tip: before submitting, **Kernel → Restart & Run All** to confirm it runs top-to-bottom.

In [1]:
import os
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
os.makedirs("charts", exist_ok=True)  # folder for saved PNGs

## Step 1 — Load Dataset & First Inspection (Day 4)
We build the raw dataset in code so the notebook is fully reproducible (in a real job this would be `pd.read_csv(...)`). Then we inspect with `head()`, `info()`, and `describe()` to spot problems.

In [2]:
data = pd.DataFrame({
    "OrderID":   [201, 202, 203, 203, 205, 206, 207, 208, 209, 210],
    "Customer":  ["  ravi", "ASHA", "imran", "imran", "Divya",
                  "KARAN", "meena ", "Sahil", "tara", "Asha"],
    "City":      ["Pune", "mumbai", "Pune", "Pune", "Delhi",
                  "Mumbai", "delhi", "Pune", "Mumbai", "mumbai"],
    "Category":  ["Electronics", "Grocery", "Electronics", "Electronics", "Grocery",
                  "Clothing", "Grocery", "Electronics", "Clothing", "Grocery"],
    "Amount":    [45000, 1200, 38000, 38000, np.nan, 2500, 999999, 52000, 3200, 1500],
    "Quantity":  [1, 8, 1, 1, 6, 2, -3, 1, 2, 5],
    "OrderDate": ["2026-05-01", "2026-05-01", "2026-05-02", "2026-05-02", "2026-05-03",
                  "2026-05-04", "2026-05-05", "2026-05-06", "2026-05-07", "2026-05-08"],
})
data.head()

,OrderID,Customer,City,Category,Amount,Quantity,OrderDate
0,201,ravi,Pune,Electronics,45000.0,1,2026-05-01
1,202,ASHA,mumbai,Grocery,1200.0,8,2026-05-01
2,203,imran,Pune,Electronics,38000.0,1,2026-05-02
3,203,imran,Pune,Electronics,38000.0,1,2026-05-02
4,205,Divya,Delhi,Grocery,NaN,6,2026-05-03


In [3]:
print("Shape:", data.shape)
data.info()

Shape: (10, 7)
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   OrderID    10 non-null     int64  
 1   Customer   10 non-null     str    
 2   City       10 non-null     str    
 3   Category   10 non-null     str    
 4   Amount     9 non-null      float64
 5   Quantity   10 non-null     int64  
 6   OrderDate  10 non-null     str    
dtypes: float64(1), int64(2), str(4)
memory usage: 692.0 bytes


In [4]:
data.describe()

,OrderID,Amount,Quantity
count,10.000000,9.000000,10.000000
mean,205.400000,131266.555556,2.400000
std,3.098387,326450.034947,3.134042
min,201.000000,1200.000000,-3.000000
25%,203.000000,2500.000000,1.000000
50%,205.500000,38000.000000,1.500000
75%,207.750000,45000.000000,4.250000
max,210.000000,999999.000000,8.000000


**Red flags spotted:** Amount max `999999` (outlier), Quantity min `-3` (impossible), 1 missing Amount (OrderID 205), OrderID 203 duplicated, mixed-case/space text, OrderDate stored as text. → The data is dirty; clean it in Step 2.

## Step 2 — Clean Dataset (Day 5)
Workflow: remove duplicate → fix types → standardize text → validate quantity → handle outlier + missing → validate.

In [5]:
df = data.copy()

# 2a. Remove duplicate on the business key (OrderID 203)
print("Duplicates before:", df.duplicated().sum())
df = df.drop_duplicates(subset=["OrderID"]).reset_index(drop=True)

# 2b. Fix type: text -> datetime
df["OrderDate"] = pd.to_datetime(df["OrderDate"])

# 2c. Standardize text
df["Customer"] = df["Customer"].str.strip().str.title()
df["City"] = df["City"].str.strip().str.title()

# 2d. Validate Quantity: -3 -> median of valid values
median_qty = df.loc[df["Quantity"] > 0, "Quantity"].median()
df["Quantity"] = df["Quantity"].astype(float)  # median may be a decimal
df.loc[df["Quantity"] < 0, "Quantity"] = median_qty

# 2e. Outlier + missing Amount: inspect with IQR, then fill with median
Q1, Q3 = df["Amount"].quantile([0.25, 0.75])
upper = Q3 + 1.5 * (Q3 - Q1)
print("IQR upper fence:", round(upper, 1))
df.loc[df["Amount"] > upper, "Amount"] = np.nan
df["Amount"] = df["Amount"].fillna(df["Amount"].median())

# 2f. Validate
assert df.isnull().sum().sum() == 0
assert df.duplicated(subset=["OrderID"]).sum() == 0
assert (df["Quantity"] > 0).all()
print("Validation passed: 0 missing, 0 duplicates, all quantities positive.")
df

Duplicates before: 1
IQR upper fence: 113500.0
Validation passed: 0 missing, 0 duplicates, all quantities positive.


,OrderID,Customer,City,Category,Amount,Quantity,OrderDate
0,201,Ravi,Pune,Electronics,45000.0,1.0,2026-05-01
1,202,Asha,Mumbai,Grocery,1200.0,8.0,2026-05-01
2,203,Imran,Pune,Electronics,38000.0,1.0,2026-05-02
3,205,Divya,Delhi,Grocery,3200.0,6.0,2026-05-03
4,206,Karan,Mumbai,Clothing,2500.0,2.0,2026-05-04
5,207,Meena,Delhi,Grocery,3200.0,2.0,2026-05-05
6,208,Sahil,Pune,Electronics,52000.0,1.0,2026-05-06
7,209,Tara,Mumbai,Clothing,3200.0,2.0,2026-05-07
8,210,Asha,Mumbai,Grocery,1500.0,5.0,2026-05-08


In [6]:
# DELIVERABLE 1: export the clean dataset
df.to_csv("freshbasket_clean.csv", index=False)
print("Saved freshbasket_clean.csv")

Saved freshbasket_clean.csv


## Step 3 — Explore the Data (Day 6)
Descriptive stats, counts, group-by revenue, and correlation.

In [7]:
print(df[["Amount", "Quantity"]].describe().round(1))
print("\nOrders per category:\n", df["Category"].value_counts(), sep="")
print("\nOrders per city:\n", df["City"].value_counts(), sep="")

        Amount  Quantity
count      9.0       9.0
mean   16644.4       3.1
std    21564.8       2.6
min     1200.0       1.0
25%     2500.0       1.0
50%     3200.0       2.0
75%    38000.0       5.0
max    52000.0       8.0

Orders per category:
Category
Grocery        4
Electronics    3
Clothing       2
Name: count, dtype: int64

Orders per city:
City
Mumbai    4
Pune      3
Delhi     2
Name: count, dtype: int64


In [8]:
rev_by_cat = df.groupby("Category")["Amount"].sum().sort_values(ascending=False)
rev_by_city = df.groupby("City")["Amount"].sum().sort_values(ascending=False)
print("Revenue by category:\n", rev_by_cat, sep="")
print("\nRevenue by city:\n", rev_by_city, sep="")
print("\nCorrelation (Quantity vs Amount):", round(df["Quantity"].corr(df["Amount"]), 2))

Revenue by category:
Category
Electronics    135000.0
Grocery          9100.0
Clothing         5700.0
Name: Amount, dtype: float64

Revenue by city:
City
Pune      135000.0
Mumbai      8400.0
Delhi       6400.0
Name: Amount, dtype: float64

Correlation (Quantity vs Amount): -0.62


**Read-out:** Electronics dominates revenue (₹135,000). Pune leads revenue while Mumbai leads order count. Correlation is **−0.62** — more quantity goes with *lower* amount (bulk Grocery vs single Electronics).

## Step 4 — Visualize (Days 7 & 8)
Six charts, each titled + labelled, saved to `charts/` at **dpi=300**. Each cell states its interpretation.

In [9]:
def save(name):
    plt.tight_layout()
    plt.savefig(f"charts/{name}", dpi=300)

# (a) Bar — revenue by category. Interpretation: Electronics dominates.
plt.figure()
sns.barplot(x=rev_by_cat.index, y=rev_by_cat.values)
plt.title("Total Revenue by Category"); plt.xlabel("Category"); plt.ylabel("Revenue (Rs.)")
save("01_revenue_by_category.png"); plt.show()

C:\Users\KOONASAISIRI\AppData\Local\Temp\ipykernel_18996\1327364128.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  save("01_revenue_by_category.png"); plt.show()


In [10]:
# (b) Bar — revenue by city. Interpretation: Pune is the revenue leader.
plt.figure()
sns.barplot(x=rev_by_city.index, y=rev_by_city.values)
plt.title("Total Revenue by City"); plt.xlabel("City"); plt.ylabel("Revenue (Rs.)")
save("02_revenue_by_city.png"); plt.show()

C:\Users\KOONASAISIRI\AppData\Local\Temp\ipykernel_18996\120912272.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  save("02_revenue_by_city.png"); plt.show()


In [11]:
# (c) Histogram — order amounts. Interpretation: most orders small; a few large ones dominate.
plt.figure()
sns.histplot(df["Amount"], bins=6, kde=True)
plt.title("Distribution of Order Amounts"); plt.xlabel("Amount (Rs.)"); plt.ylabel("Count")
save("03_amount_distribution.png"); plt.show()

C:\Users\KOONASAISIRI\AppData\Local\Temp\ipykernel_18996\559827928.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  save("03_amount_distribution.png"); plt.show()


In [12]:
# (d) Box plot — amount by category. Interpretation: Electronics highest & most variable.
plt.figure()
sns.boxplot(data=df, x="Category", y="Amount")
plt.title("Order Amount by Category"); plt.xlabel("Category"); plt.ylabel("Amount (Rs.)")
save("04_amount_by_category_box.png"); plt.show()

C:\Users\KOONASAISIRI\AppData\Local\Temp\ipykernel_18996\1446434253.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  save("04_amount_by_category_box.png"); plt.show()


In [13]:
# (e) Scatter — quantity vs amount. Interpretation: two buying patterns by category.
plt.figure()
sns.scatterplot(data=df, x="Quantity", y="Amount", hue="Category", s=120)
plt.title("Quantity vs Amount by Category"); plt.xlabel("Quantity"); plt.ylabel("Amount (Rs.)")
save("05_quantity_vs_amount.png"); plt.show()

C:\Users\KOONASAISIRI\AppData\Local\Temp\ipykernel_18996\1887790384.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  save("05_quantity_vs_amount.png"); plt.show()


In [14]:
# (f) Heatmap — correlation. Interpretation: quantifies the -0.62 relationship.
plt.figure()
sns.heatmap(df[["Amount", "Quantity"]].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
save("06_correlation_heatmap.png"); plt.show()

C:\Users\KOONASAISIRI\AppData\Local\Temp\ipykernel_18996\2940944215.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  save("06_correlation_heatmap.png"); plt.show()


## Step 5 — SQL Analysis (Day 9)
Load the clean data into in-memory SQLite and answer the business questions. (Same queries are in `sql_queries.sql`.)

In [15]:
conn = sqlite3.connect(":memory:")
df.to_sql("sales", conn, index=False, if_exists="replace")

print("Q1 — Top revenue category:")
display(pd.read_sql("SELECT Category, SUM(Amount) AS TotalRevenue FROM sales "
                    "GROUP BY Category ORDER BY TotalRevenue DESC", conn))

print("Q2 — Top city by revenue:")
display(pd.read_sql("SELECT City, SUM(Amount) AS Revenue FROM sales "
                    "GROUP BY City ORDER BY Revenue DESC", conn))

print("Q3 — High-value orders (> 10000):")
display(pd.read_sql("SELECT Customer, Category, Amount FROM sales "
                    "WHERE Amount > 10000 ORDER BY Amount DESC", conn))

print("Q4 — Pune customers:")
display(pd.read_sql("SELECT Customer, Amount FROM sales WHERE City = 'Pune'", conn))

conn.close()

Q1 — Top revenue category:


,Category,TotalRevenue
0,Electronics,135000.0
1,Grocery,9100.0
2,Clothing,5700.0


Q2 — Top city by revenue:


,City,Revenue
0,Pune,135000.0
1,Mumbai,8400.0
2,Delhi,6400.0


Q3 — High-value orders (> 10000):


,Customer,Category,Amount
0,Sahil,Electronics,52000.0
1,Ravi,Electronics,45000.0
2,Imran,Electronics,38000.0


Q4 — Pune customers:


,Customer,Amount
0,Ravi,45000.0
1,Imran,38000.0
2,Sahil,52000.0


## Step 6 — Insights (the Insight Ladder)
Observation → Finding → Insight → **Recommendation**.

| Observation | Finding | Insight | Recommendation |
|---|---|---|---|
| Electronics revenue ₹135k dwarfs others | ~90% of revenue from 3 orders | Revenue concentrated in high-value Electronics | Protect Electronics stock; avoid stockouts |
| Mumbai most orders (4), Pune most revenue (₹135k) | Pune's lead is all Electronics | Count vs revenue name different 'top' cities | Push Electronics in Pune; lift Mumbai order value |
| Grocery high qty, low amount (corr −0.62) | More units = less spend | Grocery = acquisition channel, not profit | Use Grocery to attract, upsell Electronics |
| One order was ₹999,999 (fixed) | Data-entry error existed | Input controls are weak | Add validation at data entry |

## Step 7 — Report
The full eight-section EDA report is in **`EDA_REPORT.md`**. Key takeaways: revenue is concentrated in **high-value Electronics** and the **Pune** market; **Grocery/Mumbai** drive volume not value. Act on the four recommendations above to grow revenue and data quality.